# Oseen MHM: write momentum, incompressibility and interface forms

This notebook follows **meshes → local equations → global equations → assemble
→ solve → velocity and pressure**. A user writes mixed UFL forms directly;
PyMHM supplies mesh-associated trace numbering, orientation and condensation.

We first solve an independently manufactured affine problem exactly in stable
Taylor–Hood P2/P1 local spaces. Its polynomial exactness is a correctness check,
not a convergence rate. The final section reads the separately attributed smooth
stabilized P3/P3 refinement family and plots its measured asymptotic rates.

Run with the locked `introduction` profile. No optional native cell is silently
skipped.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/710536cb72215e8d2e2d21f22e5b5c25050bad7c166dffab85cd42682d865818/oseen_variational-companion.zip"
COMPANION_SHA256 = "710536cb72215e8d2e2d21f22e5b5c25050bad7c166dffab85cd42682d865818"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("flow/oseen_variational.ipynb", directory=ROOT)


In [ ]:
from pathlib import Path
import os
import sys
ROOT = Path(os.environ.get("PYMHM_WORKSPACE", Path.cwd())).resolve()
if not (ROOT / "examples").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "examples").is_dir()), ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / "build" / "native-teaching"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Workspace:", ROOT)


## 1. Prescribe the physical solution before defining a matrix

Take viscosity one, resistance one, and constant prescribed convection
$\beta=(2,3)$. The nonconservative Oseen equation is

$$
-\Delta u+(\beta\cdot\nabla)u+u+\nabla p=f,\qquad \nabla\cdot u=0.
$$

Choose

$$
u=(1-x+2y,-2-x+y),\quad p=x+2y-0.8,\quad f=u+(5,3).
$$

The gradient of velocity is constant, its divergence vanishes, and its
Laplacian is zero. Hence $(\beta\cdot\nabla)u=(4,1)$ and $\nabla p=(1,2)$.
The physical pressure mean is 0.7. Prescribe the exact velocity on every
exterior macroface; pressure is fixed by its global mean, not by one mean per
local problem.


In [ ]:
import numpy as np
import ufl
import basix
import basix.ufl
from dolfinx import fem
from pymhm import (
    TriangleMesh, FaceSpace, SkeletonSpace, Equation, MeshHierarchy,
    LocalContext, columns, compile_form, bind_interface, bind_problem, assemble, solve,
)

def exact_velocity(x):
    """Return the solenoidal affine physical velocity."""
    return np.column_stack((1-x[:,0]+2*x[:,1], -2-x[:,0]+x[:,1]))

def exact_pressure(x):
    """Return pressure with declared physical unit-square mean 0.7."""
    return x[:,0]+2*x[:,1]-0.8

macro=TriangleMesh.unit_square(2)
skeleton=SkeletonSpace(macro,tuple(FaceSpace.uniform(1) for _ in macro.faces),2)
hierarchy=MeshHierarchy(macro,tuple(macro.submesh(i,2) for i in range(len(macro.cells))))
interface=bind_interface(skeleton,convention="normal")


taylor_hood_element=basix.ufl.mixed_element([
    basix.ufl.element("Lagrange","triangle",2,shape=(2,),lagrange_variant=basix.LagrangeVariant.equispaced),
    basix.ufl.element("Lagrange","triangle",1,lagrange_variant=basix.LagrangeVariant.equispaced),
])


## 2. Write the local mixed energy and both interface pairings

Use the skew convection form

$$
\begin{aligned}
a_K((u,p),(v,q))={}&(\nabla u,\nabla v)_K+(u,v)_K\\
&+\tfrac12[(\beta\cdot\nabla u,v)_K-(u,\beta\cdot\nabla v)_K]\\
&-(p,\nabla\cdot v)_K-(q,\nabla\cdot u)_K.
\end{aligned}
$$

Since $\nabla\cdot\beta=0$, the effective reaction is one. The normal interface
quantity is the **half-advection pseudotraction**

$$
\lambda_K=(-\nabla u+pI+\tfrac12u\otimes\beta)n_K.
$$

The interface action is $+\langle\lambda_K,v\rangle$; the global velocity pairing
is $-\langle\mu_K,u\rangle$. `trace_pairings` supplies the physical supports and
`local.equations` applies the normal orientation once. No face permutation or
orientation code is written here.

Two translations are retained through physical velocity-integral moments.
They are **coarse modes**, not null vectors when drag is positive. The local
mixed pressure has no independently imposed mean. Named fields retain the
native executed coefficient basis for later evaluation.


In [ ]:
def oseen_volume_forms(W, domain):
    """Write the common physical UFL operator, source and pressure integral."""
    u,p=ufl.TrialFunctions(W)
    v,q=ufl.TestFunctions(W)
    x=ufl.SpatialCoordinate(domain)
    beta=ufl.as_vector((2.0,3.0))
    exact=ufl.as_vector((1-x[0]+2*x[1],-2-x[0]+x[1]))
    force=exact+ufl.as_vector((5.0,3.0))
    dx=ufl.Measure("dx",domain=domain,metadata={"quadrature_degree":8})
    a=(ufl.inner(ufl.grad(u),ufl.grad(v))+ufl.inner(u,v)
       +0.5*(ufl.inner(ufl.grad(u)*beta,v)-ufl.inner(u,ufl.grad(v)*beta))
       -p*ufl.div(v)-q*ufl.div(u))*dx
    L=ufl.inner(force,v)*dx
    return a,L,q*dx

def local_oseen(local: LocalContext):
    """Declare one Taylor–Hood Oseen problem in its executed native basis."""
    binding=local.native_space(taylor_hood_element)
    W=binding.space
    u,p=ufl.TrialFunctions(W)
    v,q=ufl.TestFunctions(W)
    a,L,mean_form=oseen_volume_forms(W,binding.mesh)
    dx=ufl.Measure("dx",domain=binding.mesh,metadata={"quadrature_degree":8})
    b=local.trace_pairings(lambda phi,ds:ufl.inner(phi,v)*ds)
    c=local.trace_pairings(lambda phi,ds:-ufl.inner(phi,u)*ds,axis="rows")
    _,velocity_dofs=W.sub(0).collapse()
    translations=np.zeros((binding.size,2))
    translations[velocity_dofs]=np.tile(np.eye(2),(len(velocity_dofs)//2,1))
    local.field("velocity",binding,component=0)
    local.field("pressure",binding,component=1)
    return local.equations(
        a=a,L=L,b=b,c=c,
        coarse_basis=translations,moments=columns(v[0]*dx,v[1]*dx),
        metadata={"pressure_weights":compile_form(q*dx)},
    )


## 3. Declare the global velocity boundary equation and pressure gauge

Local `c` equations sum signed velocity continuity and exterior velocity
moments. The additional global functional has the matching minus sign.
The global physical pressure constraint is imposed **after assembly**, because
its row depends on the reconstructed local pressure responses.


In [ ]:
def global_equation(global_context):
    """Insert the prescribed velocity moments in the declared global convention."""
    boundary,_=global_context.boundary_data(exact_velocity,order=6)
    return Equation(0,-global_context.trace_load(boundary))

problem=bind_problem(hierarchy,interface,local_oseen,
                     global_equation=global_equation,retained=2)
system=assemble(problem)
pressure_weights=[record["pressure_weights"] for record in system.local_metadata]
gauge=system.mean_constraint(pressure_weights,0.7)
solution=solve(system,constraints=(gauge,))
velocity=solution.field("velocity")
pressure=solution.field("pressure")


## 4. Measure independent physical fields and original equations

The exact affine functions belong to the declared local spaces. Check both
fields at independent random physical points, and integrate the pressure mean
through its assembled physical weights. Zero errors in a polynomial patch
are roundoff controls and are not fitted as convergence slopes.


In [ ]:
maximum={"velocity":0.0,"pressure":0.0}
for vf,pf in zip(velocity,pressure,strict=True):
    bary=np.random.default_rng(127).dirichlet([1,1,1],size=40)
    vertices=vf.mesh.points[vf.mesh.cells[0]]
    points=bary@vertices
    maximum["velocity"]=max(maximum["velocity"],float(np.max(abs(vf.evaluate(points)-exact_velocity(points)))))
    maximum["pressure"]=max(maximum["pressure"],float(np.max(abs(pf.evaluate(points)-exact_pressure(points)))))
mean=sum(float(w@values) for w,values in zip(pressure_weights,solution.fields,strict=True))
assert max(maximum.values())<1e-10
np.testing.assert_allclose(mean,0.7,rtol=1e-10,atol=1e-12)
print({"maximum_physical_errors":maximum,"pressure_mean":mean,
       "original_equations_relative":solution.raw_residual})


## 5. Independently assemble a fine classical Taylor–Hood reference

The same written UFL volume operator is assembled on one globally conforming
P2/P1 space, without local condensation. Strong exterior velocity and the same
physical pressure mean make the reference a matching physical problem. Two
fine meshes independently reproduce this affine solution; errors near roundoff
are exactness controls and do not define a fitted rate.


In [ ]:
from pymhm import MultiscaleProblem
from pymhm.backends.spaces import bind_space
from pymhm.postprocessing.fields import DiscreteField,FieldDefinition

def classical_oseen(n):
    """Assemble the same mixed forms globally with strong velocity and one pressure mean."""
    fine=TriangleMesh.unit_square(n)
    binding=bind_space(fine,taylor_hood_element)
    try:
        W=binding.space
        a,L,pressure_mean=oseen_volume_forms(W,binding.mesh)
        A,F,m=compile_form(a),compile_form(L),compile_form(pressure_mean)
        V,vmap=W.sub(0).collapse()
        coordinates=V.tabulate_dof_coordinates()[:,:2]
        boundary=np.flatnonzero(np.any(np.isclose(coordinates,0)|np.isclose(coordinates,1),axis=1))
        fixed_dofs=np.asarray(vmap)[(2*boundary[:,None]+np.arange(2)).ravel()]
        values=exact_velocity(coordinates[boundary]).ravel()
        definition=MultiscaleProblem.from_global(Equation(A,F),binding.size,
            fixed=dict(zip(fixed_dofs.tolist(),values.tolist(),strict=True)),constraints=((m,0.7),))
        reference=solve(assemble(definition))
        velocity=DiscreteField(FieldDefinition("velocity",descriptor=binding.descriptor(component=0)),reference.trace)
        pressure=DiscreteField(FieldDefinition("pressure",descriptor=binding.descriptor(component=1)),reference.trace)
        points=np.random.default_rng(137).uniform(0,1,size=(100,2))
        checks={"n":n,"velocity_max":float(np.max(abs(velocity.evaluate(points)-exact_velocity(points)))),
                "pressure_max":float(np.max(abs(pressure.evaluate(points)-exact_pressure(points)))),
                "pressure_integral":float(m@reference.trace)}
        assert max(checks["velocity_max"],checks["pressure_max"])<1e-9
        np.testing.assert_allclose(checks["pressure_integral"],0.7,rtol=1e-10,atol=1e-12)
        return velocity,pressure,checks
    finally:
        binding.close()

reference_rows=[]
for n in (8,16):
    reference_velocity,reference_pressure,row=classical_oseen(n)
    reference_rows.append(row)
print("Independent conforming Taylor–Hood controls:",reference_rows)


## 5. Plot velocity and pressure

Every panel displays the actual macro mesh. Macrocells are plotted independently,
retaining their own incident field values. The analytical solution is the exact
accuracy reference for this polynomial control.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

def scalar_panel(ax, fields, exact, title, *, error=False):
    """Plot each macrocell independently, with original macro topology visible."""
    artists=[]
    for field in fields:
        fine=field.mesh
        values=field.evaluate(fine.points)
        if values.ndim==2: values=np.linalg.norm(values,axis=1)
        if exact is not None:
            truth=exact(fine.points)
            if truth.ndim==2: truth=np.linalg.norm(truth,axis=1)
            values=abs(values-truth) if error else truth
        artist=ax.tripcolor(fine.points[:,0],fine.points[:,1],fine.cells,values,shading="gouraud")
        artists.append(artist)
    lower=min(a.get_clim()[0] for a in artists);upper=max(a.get_clim()[1] for a in artists)
    for a in artists:a.set_clim(lower,upper)
    ax.add_collection(LineCollection(macro.points[macro.faces],colors="black",linewidths=0.65,alpha=0.55))
    ax.set(title=title,xlabel="x",ylabel="y",aspect="equal")
    return artists[-1]
figure,axes=plt.subplots(2,4,figsize=(15,7),layout="constrained")
for row,fields,exact,name in [(0,velocity,exact_velocity,"Velocity magnitude"),(1,pressure,exact_pressure,"Pressure")]:
    for column,title,truth,error in [(0,"Analytical",exact,False),(1,"MHM",None,False),(2,"Classical Taylor–Hood",None,False),(3,"Absolute error",exact,True)]:
        chosen=(reference_velocity,) if row==0 and column==2 else (reference_pressure,) if row==1 and column==2 else fields
        artist=scalar_panel(axes[row,column],chosen,truth,f"{title}\n{name.lower()}",error=error)
        figure.colorbar(artist,ax=axes[row,column],pad=0.025,shrink=0.85)
figure.savefig(OUTPUT/"oseen-affine-fields.png",dpi=180)
plt.show()


## 6. End with the separate smooth stabilized refinement study

The affine Taylor–Hood control above cannot produce a nonzero convergence slope.
The attributed smooth study below uses stabilized P3/P3 local fields, vector P1
traces and a structured crisscross macro mesh. Its final observed velocity L2
and pressure L2 orders approach three and two, respectively. These observations
retain the exact record and its source hash; they are not newly executed fields
from the affine control. See the theory page for convection/coercivity and
residual-test hypotheses.


In [ ]:
import json
import hashlib
from examples.tutorial_convergence import plot_method_series
record_path=ROOT/"examples/results/oseen/smooth-nu1-l1-uniform.json"
record=json.loads(record_path.read_text())
rows=record["rows"]
sizes=np.array([row["H"] for row in rows])
errors={"velocity L2":[row["velocity_l2"] for row in rows],"pressure L2":[row["pressure_l2"] for row in rows]}
rates={name:(np.log(np.array(values[:-1])/values[1:])/np.log(sizes[:-1]/sizes[1:])).tolist() for name,values in errors.items()}
series={"method":"oseen","sizes":sizes.tolist(),"errors":errors,"rates":rates,
        "expected":{"velocity L2":3.0,"pressure L2":2.0},"refinement":"Macro diameter H"}
print({"source_sha256":hashlib.sha256(record_path.read_bytes()).hexdigest(),"rates":rates})
figure=plot_method_series(series)
figure.savefig(OUTPUT/"oseen-convergence.png",dpi=180)
plt.show()


## References

- Araya, Cárcamo, Poza and Valentin (2021), *An adaptive multiscale hybrid-mixed
  method for the Oseen equations*. https://doi.org/10.1007/s10444-020-09833-8
- The smooth refinement uses the explicitly identified PyMHM mesh and spaces;
  it does not reconstruct unidentified historical adaptive meshes.
